# Notebook 44 — Sensitivity of the security metrics to the alert-family mapping

The cross-family substitution rates in Section 7 depend on an author-chosen mapping of 34 classes to eight alert
families. This notebook recomputes the recipe-conditional evaluation of Notebook 39b under three alternative mappings
and reports which conclusions are invariant. Exact-class misattribution and attack-to-benign rates do not depend on
the mapping and are unchanged by construction.

**Mappings.** M0: the original eight families. M1: DoS and DDoS merged into one denial-of-service family.
M2: web-application and credential attacks merged into one application-attack family. M3: a coarse four-family
mapping (benign; volumetric = DoS + DDoS + Mirai; probing = reconnaissance + spoofing/MITM; application = web +
credential + backdoor/upload).

**Gate (stated before running).** The recipe-conditional conclusion is mapping-invariant if, under every mapping,
the cross-family rate on identical blind-spot traffic is at least twice as high for held-out default-pruned detectors
as for held-out first-layer-protected detectors, and the ordering default > {global, protected, dense} holds.
Inference only on saved checkpoints; GPU runtime for speed.

In [ ]:
# --- Colab bootstrap ---
try:
    from google.colab import drive; drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
except Exception:
    REPO = '.'
import os, sys, copy, json as _json
os.chdir(REPO); sys.path.insert(0, REPO)
import numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.utils.prune as prune
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from src.config import CFG, PATHS, set_all_seeds
from src.data import load_raw, clean, temporal_within_capture_split
from src import train as TR, models as M, explain as EXP, mitigate
from src.comnet_audit import assign_validation_tiers, calibration_summary, environment_record, write_json
from src.train import load_anchor, predict, per_class_recall_table, feature_columns

assert torch.cuda.is_available(), 'switch to a GPU runtime first'
DEVICE = TR.DEVICE
DATASET = 'ciciot2023'
SEEDS = list(CFG['seeds']); ANCHOR = int(CFG['anchor_seed'])
OUT = PATHS.tables('comnet')
PRACTICAL_LOSS = 0.10
from itertools import combinations
RECIPES = {'default_layerwise80': 'prune80_paired', 'protect_conv0': 'layerwise80_protect_conv0_paired', 'global80': 'global80_paired'}
DENSE_OK = 0.50     # attacker-side dense recall at or above this: 'baseline-detectable'
PRUNED_BAD = 0.20   # attacker-side default-pruned recall below this: 'blind spot'

ARCH = 'cnn1d'; ARCH_KW = {'channels': (64, 128)}
print('recipes:', list(RECIPES), '| blind spot: dense >=', DENSE_OK, 'and pruned <', PRUNED_BAD)

In [ ]:
df = clean(load_raw(DATASET, subsample=True, seed=ANCHOR), DATASET)
splits = temporal_within_capture_split(df, seed=ANCHOR)
feat_cols = feature_columns(df)
print(f'{len(df):,} rows | {df.label.nunique()} classes')

In [ ]:
# Per-flow test predictions for every (recipe, seed), plus the dense baselines
fam_map = pd.read_csv(OUT / 'ciciot2023_alert_family_mapping.csv').set_index('fine_label')['alert_family'].to_dict()
def load_pruned(cell, seed, le):
    m = M.build(ARCH, len(feat_cols), len(le.classes_), **ARCH_KW).to(DEVICE)
    ck = torch.load(PATHS.model(DATASET, ARCH, cell, seed), map_location=DEVICE, weights_only=False)
    m.load_state_dict(ck['state_dict'] if isinstance(ck, dict) and 'state_dict' in ck else ck); return m.eval()

preds = {}   # (recipe, seed) -> (y_true, y_pred) as class-index arrays
for seed in SEEDS:
    m0, le, scaler, _ = load_anchor(DATASET, ARCH, 'M0_paired', seed, arch_kwargs=ARCH_KW)
    yt, yp, _ = predict(m0, df, splits, le, scaler, feat_cols, which='test'); preds[('dense', seed)] = (np.asarray(yt), np.asarray(yp))
    for recipe, cell in RECIPES.items():
        mp = load_pruned(cell, seed, le)
        yt, yp, _ = predict(mp, df, splits, le, scaler, feat_cols, which='test'); preds[(recipe, seed)] = (np.asarray(yt), np.asarray(yp))
    print(f'seed {seed}: predictions collected')
classes = list(le.classes_); C = len(classes)
benign_idx = int(np.where(np.array(classes) == 'BenignTraffic')[0][0])
family_of = np.array([fam_map[c] for c in classes])
attack_idx = [i for i in range(C) if i != benign_idx]
print(f'{C} classes, benign index {benign_idx}, {len(attack_idx)} attack classes')

In [ ]:
# Alternative mappings built from the original family labels
base = {c: fam_map[c] for c in classes}
def remap(rules):
    return {c: rules.get(f, f) for c, f in base.items()}
MAPPINGS = {
 'M0_original': dict(base),
 'M1_dos_ddos_merged': remap({'dos': 'denial_of_service', 'ddos': 'denial_of_service'}),
 'M2_web_credential_merged': remap({'web_application': 'application_attack', 'credential_attack': 'application_attack'}),
 'M3_coarse_four': remap({'dos': 'volumetric', 'ddos': 'volumetric', 'malware_botnet': 'volumetric', 'reconnaissance': 'probing', 'spoofing_mitm': 'probing',
                         'web_application': 'application', 'credential_attack': 'application'}),
}
# M3 refinement: Backdoor_Malware and Uploading_Attack are application-style, Mirai floods are volumetric
for c in ('Backdoor_Malware', 'Uploading_Attack'): MAPPINGS['M3_coarse_four'][c] = 'application'
for name, mp in MAPPINGS.items(): print(f'{name}: {len(set(mp.values()))} families')

In [ ]:
# Recipe-conditional evaluation under each mapping (blind-spot definition and folds identical to Notebook 39b)
def per_class_recall(yt, yp):
    return np.array([(yp[yt == c] == c).mean() if (yt == c).sum() else np.nan for c in range(C)])
recall = {k: per_class_recall(*v) for k, v in preds.items()}
def attacker_sets(attacker_seeds):
    r_dense = np.nanmean([recall[('dense', s)] for s in attacker_seeds], axis=0); r_pruned = np.nanmean([recall[('default_layerwise80', s)] for s in attacker_seeds], axis=0)
    detectable = [c for c in attack_idx if r_dense[c] >= DENSE_OK]; return detectable, [c for c in detectable if r_pruned[c] < PRUNED_BAD]
def metrics(yt, yp, chosen, fam):
    m = np.isin(yt, chosen); t, p = yt[m], yp[m]
    if len(t) == 0: return None
    to_benign = (p == benign_idx); cross = (fam[p] != fam[t]) & ~to_benign
    return {'misattribution_rate': float((p != t).mean()), 'attack_to_benign_rate': float(to_benign.mean()), 'cross_family_rate': float(cross.mean()), 'wrong_family_or_benign_rate': float((cross | to_benign).mean())}
rows = []
for mname, mp in MAPPINGS.items():
    fam = np.array([mp[c] for c in classes])
    for attacker_seeds in combinations(SEEDS, 3):
        held_out = [s for s in SEEDS if s not in attacker_seeds]; detectable, blind = attacker_sets(attacker_seeds)
        for recipe in list(RECIPES) + ['dense']:
            for seed in held_out:
                yt, yp = preds[(recipe, seed)]
                per = [metrics(yt, yp, [c], fam) for c in blind]; per = [d for d in per if d]
                if not per: continue
                rows.append({'mapping': mname, 'attacker_seeds': str(attacker_seeds), 'held_out_seed': seed, 'recipe': recipe, **{k: float(np.mean([d[k] for d in per])) for k in per[0]}})
fold = pd.DataFrame(rows); fold.to_csv(OUT / 'mapping_sensitivity_fold_results.csv', index=False)
summ = fold.groupby(['mapping', 'recipe'])[['misattribution_rate', 'attack_to_benign_rate', 'cross_family_rate', 'wrong_family_or_benign_rate']].mean().reset_index()
summ.to_csv(OUT / 'mapping_sensitivity_summary.csv', index=False); pd.set_option('display.width', 200); print(summ.round(4).to_string(index=False))

In [ ]:
# Benign flood does not depend on the mapping; gate on the cross-family conclusion under every mapping
ver = []
for mname in MAPPINGS:
    g = summ[summ.mapping == mname].set_index('recipe').cross_family_rate
    ratio = float(g['default_layerwise80'] / max(g['protect_conv0'], 1e-9))
    ordering = bool(g['default_layerwise80'] > max(g['protect_conv0'], g['global80'], g['dense']))
    ver.append({'mapping': mname, 'cross_family_default': round(float(g['default_layerwise80']), 4), 'cross_family_protected': round(float(g['protect_conv0']), 4),
                'cross_family_global': round(float(g['global80']), 4), 'cross_family_dense': round(float(g['dense']), 4), 'default_over_protected': round(ratio, 3),
                'ratio_ge_2': bool(ratio >= 2.0), 'default_worst': ordering})
verdict = pd.DataFrame(ver); print(verdict.to_string(index=False))
print('\nRecipe-conditional conclusion invariant to the family mapping:', bool(verdict.ratio_ge_2.all() and verdict.default_worst.all()))
verdict.to_csv(OUT / 'mapping_sensitivity_gate_verdict.csv', index=False)
write_json(OUT / 'mapping_sensitivity_environment.json', {'mappings': {k: sorted(set(v.values())) for k, v in MAPPINGS.items()}, 'seeds': SEEDS, 'environment': environment_record()})

In [ ]:
# --- Commit + push: main only, this notebook's own files only ---
import subprocess, shutil, glob
_b = subprocess.run(['git', 'rev-parse', '--abbrev-ref', 'HEAD'], capture_output=True, text=True).stdout.strip()
assert _b == 'main', f'checked-out branch is {_b!r}; run `git checkout main` first'
subprocess.run(['git', 'config', '--global', 'user.name', 'Md Anas Biswas'], check=True)
subprocess.run(['git', 'config', '--global', 'user.email', 'anasbiswas@gmail.com'], check=True)
cred = '/content/drive/MyDrive/IoT_Trust_Research/.git-credentials'
if os.path.exists(cred):
    shutil.copy(cred, '/root/.git-credentials'); subprocess.run(['git', 'config', '--global', 'credential.helper', 'store'], check=True)
_own = 'notebooks/44_family_mapping_sensitivity.ipynb'
if os.path.exists(_own):
    d = _json.load(open(_own))
    for c in d.get('cells', []):
        if c.get('cell_type') == 'code': c['outputs'] = []; c['execution_count'] = None
    _json.dump(d, open(_own, 'w'), indent=1)
subprocess.run(['git', 'add', _own] + glob.glob('results/tables/comnet/mapping_sensitivity_*'), check=True)
r = subprocess.run(['git', 'commit', '-m', 'notebook 44: sensitivity of the recipe-conditional security result to three alternative alert-family mappings'], capture_output=True, text=True)
print(r.stdout or r.stderr)
print(subprocess.run(['git', 'push'], capture_output=True, text=True).stderr or 'pushed')
print(subprocess.run(['git', 'log', '--oneline', '-2'], capture_output=True, text=True).stdout)